# Complete Databricks: Zero to Expert
**How to use:** (1) run `Databricks_Environment_Setup.ipynb` first; (2) run the Setup cells below; (3) run modules top to bottom. Every risky cell is wrapped in `guard(...)`, so an unsupported feature records SKIPPED/FAILED and the notebook keeps going. The final cell prints a pass/fail/skip summary.

**Accuracy note:** written from training knowledge, not re-verified against live docs. Features marked **[verify]** are preview or fast-changing. Check the Databricks release notes for your runtime and cloud.

**Table of contents**
Level 1: M1 Basics, M2 dbutils | Level 2: M3 PySpark, M4 Spark SQL, M5 Unity Catalog, M6 Delta Lake, M7 Ingestion |
Level 3: M8 Streaming, M9 Data engineering pipelines, M10 Jobs, M12 Performance, M14 Security |
Level 4: M11 REST/CLI/SDK, M13 ML/AI, M15 Developer techniques | Level 5: M16 Architecture |
Appendices: interview Q&A, glossary, command index, compatibility pointers, final validation.

**Notebook execution contexts:** Python/SQL/Scala/R magics run in a notebook. CLI, Terraform, pytest runners and bundles run in a terminal or CI, shown here as text or files written to a volume.

## Setup (run first)

In [ ]:
import sys, os, json, time, random, datetime as dt
from contextlib import contextmanager
from pyspark.sql import functions as F, types as T, Window

C = "databricks_learning"
B, S, G, SB, AD = (f"{C}.{x}" for x in ["bronze", "silver", "gold", "sandbox", "advanced"])
VOL = f"/Volumes/{C}/sandbox/learning_volume"
RESULTS = []

@contextmanager
def guard(name):
    """Record PASSED/FAILED; swallow errors so later cells still run."""
    try:
        yield
        RESULTS.append((name, "PASSED", ""))
    except Exception as e:
        msg = (str(e).strip().splitlines() or [type(e).__name__])[0][:250]
        RESULTS.append((name, "FAILED", msg)); print(f"[FAILED] {name}: {msg}")

def skip(name, why):
    RESULTS.append((name, "SKIPPED", why)); print(f"[SKIPPED] {name}: {why}")

def conf(k):
    try: return spark.conf.get(k)
    except Exception: return None

ME = spark.sql("SELECT current_user()").first()[0]
need = [f"{B}.customers", f"{B}.products", f"{B}.orders", f"{B}.payments", f"{B}.events"]
missing = [t for t in need if not spark.catalog.tableExists(t)]
assert not missing, f"Run Databricks_Environment_Setup.ipynb first. Missing: {missing}"
spark.sql(f"USE CATALOG {C}")
print("ready as", ME, "| spark", spark.version)

# LEVEL 1: ABSOLUTE BEGINNER

## Module 1: Notebook basics and Python

**Magic commands:** `%python`, `%sql`, `%scala`, `%r` switch language per cell; `%md` renders Markdown; `%sh` runs shell on the driver only; `%pip install pkg` installs notebook-scoped libraries (restarts Python state, put it at the top; on serverless use the Environment panel for repeatability); `%run ./other_notebook` executes another notebook in the current session (shares variables).
Scala and R cells need compute that supports them (not serverless Scala/R in all workspaces) **[verify]**.

In [ ]:
# Variables, types, control flow, functions, classes, exceptions, files
name, n, pi, flag = "Databricks", 3, 3.14159, True
nums = [1, 2, 3]; person = {"id": 1, "city": "Hyderabad"}; uniq = {1, 2, 2}; tup = (1, "a")
print(type(name), type(n), type(pi), type(flag), nums, person, uniq, tup)

for i in range(3):
    print("loop", i)
print("even" if n % 2 == 0 else "odd")

def add(a, b=0):
    return a + b

class Order:
    def __init__(self, oid, amount): self.oid, self.amount = oid, amount
    def total_with_tax(self, rate=0.18): return round(self.amount * (1 + rate), 2)
print(add(2, 3), Order(1, 100).total_with_tax())

try:
    1 / 0
except ZeroDivisionError as e:
    print("caught:", e)
finally:
    print("always runs")

import math, collections
with open("/tmp/hello.txt", "w") as f: f.write("hello\n")
print(open("/tmp/hello.txt").read(), math.sqrt(16))

In [ ]:
%sql
SELECT current_catalog() AS catalog, current_user() AS me, current_date() AS today

In [ ]:
%sh
echo "driver shell works"; python --version; ls /tmp | head -3

## Module 2: Databricks Utilities (dbutils)

### dbutils.fs
**Purpose:** File ops on DBFS, volumes and cloud paths
**Syntax:** `dbutils.fs.ls|mkdirs|put|head|cp|mv|rm|mount`
**Prerequisites / permissions:** READ/WRITE VOLUME on the volume
**Common errors:** Path not found; DBFS root disabled; mounts unsupported with UC (use volumes/external locations)
**Performance / security:** Use volumes, not mounts. `rm(path, True)` is recursive.
**Cleanup:** dbutils.fs.rm(path, True)
**Expected output:** shown under the next cell after you run it.

In [ ]:
with guard("dbutils.fs"):
    d = f"{VOL}/dbutils_demo"
    dbutils.fs.mkdirs(d)
    dbutils.fs.put(f"{d}/a.txt", "line1\nline2\n", True)
    dbutils.fs.cp(f"{d}/a.txt", f"{d}/b.txt")
    dbutils.fs.mv(f"{d}/b.txt", f"{d}/c.txt")
    print([f.name for f in dbutils.fs.ls(d)])
    print(dbutils.fs.head(f"{d}/a.txt"))
    dbutils.fs.rm(f"{d}/c.txt")

### dbutils.widgets
**Purpose:** Notebook parameters (also used by job parameters)
**Syntax:** `dbutils.widgets.text|dropdown|combobox|multiselect|get|remove|removeAll`
**Prerequisites / permissions:** none
**Common errors:** get() on a missing widget raises
**Performance / security:** Widgets are strings; cast them.
**Cleanup:** dbutils.widgets.removeAll()
**Expected output:** shown under the next cell after you run it.

In [ ]:
with guard("dbutils.widgets"):
    dbutils.widgets.text("env", "dev", "Environment")
    dbutils.widgets.dropdown("mode", "full", ["full", "incremental"], "Load mode")
    print(dbutils.widgets.get("env"), dbutils.widgets.get("mode"))
    dbutils.widgets.removeAll()

### dbutils.secrets
**Purpose:** Read secrets without exposing them
**Syntax:** `dbutils.secrets.listScopes|list(scope)|get(scope,key)|getBytes`
**Prerequisites / permissions:** READ on the scope (ACL)
**Common errors:** Scope does not exist; output shows [REDACTED]
**Performance / security:** Never print or log secrets; create scopes via CLI/API only.
**Cleanup:** databricks secrets delete-scope <scope>
**Expected output:** shown under the next cell after you run it.

In [ ]:
with guard("dbutils.secrets"):
    scopes = [s.name for s in dbutils.secrets.listScopes()]
    print("scopes:", scopes)
    if scopes:
        print([k.key for k in dbutils.secrets.list(scopes[0])])

**Other utilities:** `dbutils.notebook.run(path, timeout, args)` runs a child notebook and returns its `dbutils.notebook.exit(value)` string; `dbutils.notebook.exit("x")` ends the current one (not executed here since it would stop this notebook). `dbutils.jobs.taskValues.set/get` passes values between job tasks (only meaningful inside a job). `dbutils.credentials` (service credentials) and `dbutils.data.summarize(df)` need supported runtimes **[verify]**. `dbutils.library` is removed/deprecated in favor of `%pip`. `dbutils.help()` lists utilities.

In [ ]:
with guard("dbutils.data.summarize"):
    dbutils.data.summarize(spark.table(f"{B}.orders"))
with guard("dbutils.help"):
    dbutils.help()

**Exercises:** (1) write a widget-driven notebook that filters orders by status; (2) copy a file between two volume folders and list it; (3) fetch a secret and show its length only.

# LEVEL 2: INTERMEDIATE

## Module 3: Apache Spark and PySpark

**Concepts:** *Transformations* (select, filter, join, groupBy) are lazy and build a plan. *Actions* (count, show, collect, write) trigger a job. A job splits into **stages** at shuffle boundaries (wide transformations: join, groupBy, distinct, repartition); a stage runs one **task** per partition. Narrow transformations (filter, select, map) need no shuffle. Catalyst optimizes the logical plan; Tungsten handles memory/codegen.

In [ ]:
with guard("dataframe basics"):
    orders = spark.table(f"{B}.orders"); customers = spark.table(f"{B}.customers"); products = spark.table(f"{B}.products")
    orders.printSchema()
    orders.select("order_id", "amount").filter(F.col("amount") > 100).orderBy(F.desc("amount")).show(3)
    print(orders.count(), orders.columns, orders.dtypes[:2])
    orders.describe("amount").show(); orders.summary("count", "mean", "50%").show()

In [ ]:
with guard("lazy eval and plans"):
    q = orders.filter("status = 'PAID'").groupBy("customer_id").agg(F.sum("amount").alias("total"))
    q.explain()               # nothing executed yet
    q.explain("formatted")
    print("partitions:", q.rdd.getNumPartitions())

In [ ]:
with guard("column ops"):
    df = (orders.withColumn("tax", F.round(F.col("amount") * 0.18, 2))
                .withColumn("size", F.when(F.col("amount") > 500, "big").when(F.col("amount") > 100, "mid").otherwise("small"))
                .withColumnRenamed("amount", "amt").drop("quantity")
                .withColumn("order_date", F.to_date("order_ts")))
    df.show(3)
    df.selectExpr("order_id", "amt * 2 AS double_amt", "cast(order_id AS string) AS id_s").show(3)

In [ ]:
with guard("joins"):
    j = orders.join(customers, "customer_id", "inner").join(F.broadcast(products), "product_id", "left")
    j.select("order_id", customers["name"].alias("customer"), products["name"].alias("product"), "amount").show(3)
    orders.join(customers, "customer_id", "left_anti").count()
    orders.join(customers, "customer_id", "left_semi").count()
    orders.crossJoin(F.broadcast(products.limit(2))).count()
    # multi-key join
    orders.alias("a").join(orders.alias("b"), ["customer_id", "product_id"]).count()
    # union by name vs position
    orders.unionByName(orders).count()

In [ ]:
with guard("aggregations and windows"):
    orders.groupBy("status").agg(F.count("*").alias("n"), F.avg("amount").alias("avg"),
        F.approx_count_distinct("customer_id").alias("cust_approx"), F.countDistinct("customer_id").alias("cust_exact")).show()
    w = Window.partitionBy("customer_id").orderBy(F.desc("order_ts"))
    (orders.withColumn("rn", F.row_number().over(w)).withColumn("prev_amt", F.lag("amount").over(w))
           .withColumn("running", F.sum("amount").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
           .filter("rn <= 2").show(4))

In [ ]:
with guard("complex types"):
    nested = (orders.groupBy("customer_id").agg(F.collect_list("order_id").alias("order_ids"),
                F.collect_set("status").alias("statuses"))
              .withColumn("info", F.struct(F.size("order_ids").alias("n"), F.lit("x").alias("tag")))
              .withColumn("m", F.create_map(F.lit("n"), F.size("order_ids").cast("string"))))
    nested.select("customer_id", F.explode("order_ids").alias("order_id"), "info.n", F.element_at("m", "n")).show(3)
    nested.select(F.posexplode_outer("statuses")).show(3)
    js = F.to_json(F.struct("order_id", "status"))
    parsed = orders.select(F.from_json(js, "order_id INT, status STRING").alias("p")).select("p.*")
    parsed.show(2)

In [ ]:
with guard("pivot and unpivot"):
    pv = orders.groupBy("customer_id").pivot("status", ["PAID", "PENDING", "CANCELLED"]).agg(F.sum("amount"))
    pv.show(3)
    pv.unpivot("customer_id", ["PAID", "PENDING", "CANCELLED"], "status", "amount").show(3)

In [ ]:
with guard("udf and pandas udf"):
    @F.udf("string")
    def bucket(a): return "hi" if a and a > 250 else "lo"          # row-at-a-time, slow
    @F.pandas_udf("double")
    def with_tax(s):                                                  # vectorized via Arrow
        return s * 1.18
    orders.select(bucket("amount").alias("b"), with_tax("amount").alias("t")).show(3)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [ ]:
with guard("partitions, cache, config"):
    r = orders.repartition(8, "customer_id"); print("repartition:", r.rdd.getNumPartitions())
    print("coalesce:", r.coalesce(2).rdd.getNumPartitions())
    orders.cache(); orders.count(); orders.unpersist()
    print("shuffle partitions:", conf("spark.sql.shuffle.partitions"))

In [ ]:
# RDD / broadcast variables / accumulators need SparkContext: not available on serverless or standard access mode
try:
    sc = spark.sparkContext
    rdd = sc.parallelize(range(10), 2)
    acc = sc.accumulator(0); bc = sc.broadcast({"a": 1})
    rdd.foreach(lambda x: acc.add(x))
    print(rdd.map(lambda x: x * 2).take(3), acc.value, bc.value)
    RESULTS.append(("rdd/broadcast/accumulator", "PASSED", ""))
except Exception as e:
    skip("rdd/broadcast/accumulator", "SparkContext not available on this compute (use dedicated/classic compute)")

In [ ]:
skip("dataframe.checkpoint", "needs sc.setCheckpointDir (classic compute); persist to a Delta table instead")

## Module 4: Spark SQL reference with examples

In [ ]:
with guard("sql ddl dml"):
    spark.sql(f"CREATE OR REPLACE TABLE {SB}.demo (id INT, name STRING, qty INT) USING DELTA")
    spark.sql(f"INSERT INTO {SB}.demo VALUES (1,'a',5),(2,'b',0),(3,'c',9)")
    spark.sql(f"UPDATE {SB}.demo SET qty = qty + 1 WHERE id = 2")
    spark.sql(f"DELETE FROM {SB}.demo WHERE id = 3")
    spark.sql(f"ALTER TABLE {SB}.demo ADD COLUMNS (note STRING)")
    spark.sql(f"INSERT OVERWRITE {SB}.demo SELECT id, name, qty, 'x' FROM {SB}.demo")
    spark.sql(f"SELECT * FROM {SB}.demo").show()
    spark.sql(f"TRUNCATE TABLE {SB}.demo")

In [ ]:
%sql
WITH paid AS (SELECT * FROM databricks_learning.bronze.orders WHERE status = 'PAID'),
     per_c AS (SELECT customer_id, SUM(amount) AS total, COUNT(*) AS n FROM paid GROUP BY customer_id HAVING COUNT(*) >= 2)
SELECT c.city, COUNT(*) AS customers, ROUND(AVG(total),2) AS avg_total,
       RANK() OVER (ORDER BY SUM(total) DESC) AS city_rank
FROM per_c JOIN databricks_learning.bronze.customers c USING (customer_id)
GROUP BY c.city ORDER BY city_rank

In [ ]:
with guard("sql set ops and misc"):
    spark.sql(f"SELECT customer_id FROM {B}.orders INTERSECT SELECT customer_id FROM {B}.customers").count()
    spark.sql(f"SELECT customer_id FROM {B}.customers EXCEPT SELECT customer_id FROM {B}.orders").show(3)
    spark.sql(f"SELECT customer_id FROM {B}.orders UNION SELECT customer_id FROM {B}.customers").count()
    spark.sql(f"SELECT order_id, CASE WHEN amount>300 THEN 'hi' ELSE 'lo' END b, CAST(amount AS INT) a FROM {B}.orders LIMIT 3").show()
    spark.sql(f"EXPLAIN FORMATTED SELECT * FROM {B}.orders WHERE status='PAID'").show(truncate=False)
    spark.sql(f"ANALYZE TABLE {B}.orders COMPUTE STATISTICS FOR ALL COLUMNS")
    spark.sql(f"CACHE TABLE {B}.customers"); spark.sql(f"UNCACHE TABLE {B}.customers")
    spark.sql(f"REFRESH TABLE {B}.orders")
    spark.sql("SET spark.sql.shuffle.partitions=64"); spark.sql("RESET spark.sql.shuffle.partitions")

**DCL/TCL notes:** `GRANT/REVOKE/DENY` appear in Module 5. Delta provides per-table ACID transactions; there is no `BEGIN/COMMIT` across tables in standard Databricks SQL (multi-statement transactions are limited/preview **[verify]**). `DENY` is available in Unity Catalog on supported securables **[verify]** and was not in all earlier releases.

## Module 5: Unity Catalog

In [ ]:
with guard("uc inspect"):
    spark.sql("SELECT current_metastore()").show(truncate=False)
    for q in [f"SHOW SCHEMAS IN {C}", f"SHOW TABLES IN {B}", f"SHOW VOLUMES IN {SB}",
              f"DESCRIBE TABLE EXTENDED {B}.orders", f"SHOW GRANTS ON SCHEMA {B}"]:
        print(">>", q); spark.sql(q).show(20, truncate=False)

In [ ]:
with guard("views and functions"):
    spark.sql(f"CREATE OR REPLACE VIEW {G}.v_paid_orders AS SELECT * FROM {B}.orders WHERE status='PAID'")
    spark.sql(f"CREATE OR REPLACE FUNCTION {G}.fn_tax(a DOUBLE) RETURNS DOUBLE RETURN ROUND(a*0.18,2)")
    spark.sql(f"SELECT order_id, {G}.fn_tax(amount) t FROM {G}.v_paid_orders LIMIT 3").show()
    spark.sql(f"COMMENT ON TABLE {B}.orders IS 'raw orders'")
    spark.sql(f"ALTER TABLE {B}.orders SET TAGS ('domain'='sales')")
    spark.sql(f"SELECT table_name, table_type FROM {C}.information_schema.tables WHERE table_schema='bronze'").show()

In [ ]:
with guard("row filter and column mask"):
    spark.sql(f"CREATE OR REPLACE TABLE {SB}.customers_secure AS SELECT * FROM {B}.customers")
    spark.sql(f"CREATE OR REPLACE FUNCTION {SB}.rf_city(city STRING) RETURN is_account_group_member('admins') OR city <> 'Delhi'")
    spark.sql(f"CREATE OR REPLACE FUNCTION {SB}.mask_email(email STRING) RETURN CASE WHEN is_account_group_member('admins') THEN email ELSE '*****' END")
    spark.sql(f"ALTER TABLE {SB}.customers_secure SET ROW FILTER {SB}.rf_city ON (city)")
    spark.sql(f"ALTER TABLE {SB}.customers_secure ALTER COLUMN email SET MASK {SB}.mask_email")
    spark.sql(f"SELECT city, email FROM {SB}.customers_secure LIMIT 3").show()
    spark.sql(f"ALTER TABLE {SB}.customers_secure DROP ROW FILTER")
    spark.sql(f"ALTER TABLE {SB}.customers_secure ALTER COLUMN email DROP MASK")

**GRANT / REVOKE / DENY (owner or admin; replace `analysts`):**
```sql
GRANT USE CATALOG ON CATALOG databricks_learning TO `analysts`;
GRANT USE SCHEMA, SELECT ON SCHEMA databricks_learning.gold TO `analysts`;
REVOKE SELECT ON SCHEMA databricks_learning.gold FROM `analysts`;
SHOW GRANTS ON TABLE databricks_learning.bronze.orders;
```
Privileges inherit downward (catalog, schema, table). A principal needs `USE CATALOG` and `USE SCHEMA` plus `SELECT`.
**Not executable in a standard notebook:** metastore creation, workspace assignment (account console); storage credentials and external locations (need cloud IAM; see Setup notebook section 10); connections/foreign catalogs (`CREATE CONNECTION`, `CREATE FOREIGN CATALOG`, need a reachable source); Delta Sharing (`CREATE SHARE/RECIPIENT`, need `CREATE SHARE`/`CREATE RECIPIENT` and metastore sharing enabled); materialized views and metric views (SQL warehouse or pipeline, **[verify]**); lineage and discovery (Catalog Explorer UI); audit logs (system tables, Module 14).

## Module 6: Delta Lake

In [ ]:
with guard("delta basics"):
    t = f"{SB}.delta_demo"
    spark.sql(f"""CREATE OR REPLACE TABLE {t} (
        id BIGINT GENERATED ALWAYS AS IDENTITY, order_id INT NOT NULL, amount DOUBLE,
        ts TIMESTAMP, d DATE GENERATED ALWAYS AS (CAST(ts AS DATE)), CONSTRAINT pos CHECK (amount >= 0))
        TBLPROPERTIES (delta.enableChangeDataFeed = true, delta.enableDeletionVectors = true)""")
    spark.sql(f"INSERT INTO {t} (order_id, amount, ts) SELECT order_id, amount, order_ts FROM {B}.orders WHERE order_id <= 200")
    spark.sql(f"UPDATE {t} SET amount = amount * 1.1 WHERE order_id <= 20")
    spark.sql(f"DELETE FROM {t} WHERE order_id > 190")
    spark.sql(f"DESCRIBE HISTORY {t}").select("version", "operation", "operationMetrics").show(truncate=False)

In [ ]:
with guard("time travel and restore"):
    v = spark.sql(f"DESCRIBE HISTORY {t}").agg(F.max("version")).first()[0]
    print("now:", spark.table(t).count(), "| v1:", spark.sql(f"SELECT * FROM {t} VERSION AS OF 1").count())
    spark.sql(f"RESTORE TABLE {t} TO VERSION AS OF 1")
    print("restored:", spark.table(t).count())
    spark.sql(f"SELECT _change_type, COUNT(*) n FROM table_changes('{t}', 1) GROUP BY 1").show()

In [ ]:
with guard("constraint enforcement"):
    try:
        spark.sql(f"INSERT INTO {t} (order_id, amount, ts) VALUES (9999, -5, current_timestamp())")
        raise AssertionError("constraint did not fire")
    except AssertionError: raise
    except Exception as e: print("expected rejection:", str(e).splitlines()[0][:120])

In [ ]:
with guard("optimize vacuum liquid"):
    spark.sql(f"OPTIMIZE {t}")
    spark.sql(f"VACUUM {t} DRY RUN").show(3, truncate=False)
    lc = f"{SB}.liquid_demo"
    spark.sql(f"CREATE OR REPLACE TABLE {lc} CLUSTER BY (customer_id) AS SELECT * FROM {B}.orders")
    spark.sql(f"OPTIMIZE {lc}")
    spark.sql(f"DESCRIBE DETAIL {lc}").select("clusteringColumns", "numFiles").show()
    spark.sql(f"ALTER TABLE {lc} CLUSTER BY (status)")      # change keys; use CLUSTER BY NONE to disable

In [ ]:
with guard("merge, deltatable api, clone, schema evolution"):
    from delta.tables import DeltaTable
    upd = spark.createDataFrame([(1, 999.0), (100000, 5.0)], "order_id INT, amount DOUBLE")
    dt_ = DeltaTable.forName(spark, t)
    (dt_.alias("t").merge(upd.alias("s"), "t.order_id = s.order_id")
        .whenMatchedUpdate(set={"amount": "s.amount"})
        .whenNotMatchedInsert(values={"order_id": "s.order_id", "amount": "s.amount", "ts": "current_timestamp()"})
        .execute())
    dt_.history(1).show(truncate=False)
    spark.sql(f"CREATE OR REPLACE TABLE {SB}.delta_shallow SHALLOW CLONE {t}")
    spark.sql(f"CREATE OR REPLACE TABLE {SB}.delta_deep DEEP CLONE {t}")
    new = spark.table(B + ".orders").limit(5).withColumn("extra", F.lit("x"))
    new.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{SB}.liquid_demo")

**Internals:** `_delta_log/` holds ordered JSON commits and periodic Parquet checkpoints; readers replay the log to find the live file set. Writers use optimistic concurrency: conflicting commits raise `ConcurrentAppendException`/`ConcurrentDeleteReadException` and should be retried or avoided with partition/cluster-aligned predicates. Deletion vectors mark rows deleted without rewriting files (compaction later purges them). Streaming tables and Lakeflow Spark Declarative Pipelines (formerly Delta Live Tables) are defined in a pipeline, not run in a plain notebook:
```sql
CREATE OR REFRESH STREAMING TABLE bronze_orders AS SELECT * FROM STREAM read_files('/Volumes/databricks_learning/sandbox/learning_volume/landing/orders', format => 'json');
CREATE OR REFRESH MATERIALIZED VIEW gold_sales AS SELECT status, SUM(amount) total FROM bronze_orders GROUP BY status;
```
Python uses `from pyspark import pipelines as dp` with `@dp.table` / `@dp.materialized_view` (older: `import dlt`, `@dlt.table`) **[verify]**.

## Module 7: Data ingestion and file formats

In [ ]:
with guard("write and read formats"):
    src = spark.table(f"{B}.orders").limit(200)
    base = f"{VOL}/formats"
    for fmt in ["csv", "json", "parquet", "orc", "avro"]:
        w = src.write.mode("overwrite").format(fmt)
        w = w.option("header", "true") if fmt == "csv" else w
        w.save(f"{base}/{fmt}")
    dbutils.fs.put(f"{base}/text/a.txt", "x\ny\nz\n", True)
    print("csv    ", spark.read.option("header", "true").option("inferSchema", "true").csv(f"{base}/csv").count())
    print("json   ", spark.read.json(f"{base}/json").count())
    print("parquet", spark.read.parquet(f"{base}/parquet").count())
    print("orc    ", spark.read.orc(f"{base}/orc").count())
    print("avro   ", spark.read.format("avro").load(f"{base}/avro").count())
    print("text   ", spark.read.text(f"{base}/text").count(), "| wholetext:", spark.read.option("wholetext", "true").text(f"{base}/text").count())
    spark.read.format("binaryFile").load(f"{base}/parquet").select("path", "length").show(2, truncate=False)

In [ ]:
with guard("explicit schema, bad records, rescue"):
    dbutils.fs.put(f"{VOL}/formats/bad/b.csv", "id,amt\n1,10\nx,notnum\n3,30\n", True)
    sch = "id INT, amt DOUBLE, _corrupt STRING"
    spark.read.schema(sch).option("header", "true").option("mode", "PERMISSIVE") \
        .option("columnNameOfCorruptRecord", "_corrupt").csv(f"{VOL}/formats/bad").show(truncate=False)
    spark.read.schema("id INT, amt DOUBLE").option("header", "true").option("mode", "DROPMALFORMED").csv(f"{VOL}/formats/bad").show()
    spark.read.schema("id INT, amt DOUBLE").option("header", "true").option("rescuedDataColumn", "_rescued").csv(f"{VOL}/formats/bad").show(truncate=False)

In [ ]:
with guard("read_files"):
    spark.sql(f"SELECT * FROM read_files('{VOL}/raw/orders_sample.csv', format => 'csv', header => true) LIMIT 3").show()
    spark.sql(f"SELECT * FROM read_files('{VOL}/raw/customers_sample.json', format => 'json') LIMIT 3").show()

In [ ]:
with guard("copy into"):
    spark.sql(f"CREATE OR REPLACE TABLE {SB}.copy_target (order_id INT, customer_id INT, product_id INT, quantity INT, amount DOUBLE, status STRING, order_ts TIMESTAMP)")
    q = f"""COPY INTO {SB}.copy_target FROM '{VOL}/raw/orders_sample.csv'
            FILEFORMAT = CSV FORMAT_OPTIONS ('header'='true','inferSchema'='true') COPY_OPTIONS ('mergeSchema'='true')"""
    spark.sql(q).show()
    spark.sql(q).show()        # idempotent: second run loads 0 new rows
    # VALIDATE ALL dry-runs: COPY INTO ... VALIDATE ALL ; FORCE = 'true' reloads files

In [ ]:
with guard("auto loader (availableNow)"):
    land = f"{VOL}/landing/orders"; dbutils.fs.mkdirs(land)
    spark.table(f"{B}.orders").limit(100).write.mode("overwrite").json(land)
    q = (spark.readStream.format("cloudFiles").option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", f"{VOL}/_schemas/orders")
         .option("cloudFiles.inferColumnTypes", "true").option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option("cloudFiles.rescuedDataColumn", "_rescued_data").load(land)
         .writeStream.option("checkpointLocation", f"{VOL}/_ckpt/autoloader_orders")
         .option("mergeSchema", "true").trigger(availableNow=True).toTable(f"{SB}.autoloader_orders"))
    q.awaitTermination()
    print(spark.table(f"{SB}.autoloader_orders").count())

**Auto Loader modes:** *directory listing* (default, no cloud setup) vs *file notification* (`cloudFiles.useNotifications=true`, needs cloud permissions to create queue/topic; better at huge directories). Schema evolution modes: `addNewColumns` (default; stream stops once then picks up new columns on restart), `rescue`, `failOnNewColumns`, `none`. Exactly-once holds for the file-to-sink path via checkpoint, not across overwritten files (`cloudFiles.allowOverwrites`) or non-idempotent sinks.
**JDBC (needs a reachable database; not executed):**
```python
df = (spark.read.format("jdbc").option("url", "jdbc:postgresql://host:5432/db")
      .option("dbtable", "public.orders").option("user", dbutils.secrets.get("scope","user"))
      .option("password", dbutils.secrets.get("scope","pw"))
      .option("partitionColumn","id").option("lowerBound","1").option("upperBound","1000000")
      .option("numPartitions","8").option("fetchsize","10000").load())
df.write.mode("append").saveAsTable("databricks_learning.bronze.pg_orders")
```
Lakehouse Federation (`CREATE CONNECTION` + `CREATE FOREIGN CATALOG`) is the governed alternative. XML: `spark.read.format("xml").option("rowTag","row").load(path)` on recent runtimes **[verify]**.
**Lakeflow Connect:** configured in the UI/API (managed connectors for SaaS and databases) and lands data into UC tables. Not executable from a notebook.

# LEVEL 3: ADVANCED

## Module 8: Structured Streaming

In [ ]:
with guard("streaming delta to delta"):
    src_t = f"{SB}.stream_src"; dst_t = f"{SB}.stream_agg"
    spark.sql(f"CREATE OR REPLACE TABLE {src_t} AS SELECT * FROM {B}.events")
    ev = (spark.readStream.table(src_t).withWatermark("event_ts", "10 minutes")
          .groupBy(F.window("event_ts", "1 day"), "event_type").count())
    q = (ev.writeStream.outputMode("append").option("checkpointLocation", f"{VOL}/_ckpt/stream_agg")
         .trigger(availableNow=True).toTable(dst_t))
    q.awaitTermination()
    print("lastProgress:", (q.lastProgress or {}).get("numInputRows"), "| status:", q.status)
    spark.table(dst_t).show(3)

In [ ]:
with guard("foreachBatch, dedup, stream-static join"):
    cust = spark.table(f"{B}.customers")
    def upsert(batch_df, batch_id):
        j = batch_df.dropDuplicates(["event_id"]).join(cust, "customer_id")
        j.write.mode("append").saveAsTable(f"{SB}.stream_joined")
    q = (spark.readStream.table(src_t).writeStream.foreachBatch(upsert)
         .option("checkpointLocation", f"{VOL}/_ckpt/stream_fb").trigger(availableNow=True).start())
    q.awaitTermination()
    print([s.name for s in spark.streams.active])

**Notes:** Triggers: `processingTime="1 minute"`, `availableNow=True` (preferred batch-like incremental; `once=True` is deprecated). Output modes: `append`, `update`, `complete`. Watermark = how late data is tolerated; state older than the watermark is dropped. Stream-stream joins need watermarks on both sides plus a time-range condition. `dropDuplicatesWithinWatermark` bounds dedup state. Recovery: restart the query with the **same checkpoint**; changing the query shape may require a new checkpoint. Monitor with `query.status`, `lastProgress`, `recentProgress`, `query.stop()`.

## Module 9: Data engineering pipelines (medallion, CDC, SCD, quality)

In [ ]:
with guard("bronze to silver to gold"):
    silver = (spark.table(f"{B}.orders").dropDuplicates(["order_id"]).na.drop(subset=["order_id", "customer_id"])
              .withColumn("status", F.upper(F.trim("status"))).withColumn("amount", F.col("amount").cast("decimal(12,2)"))
              .withColumn("order_date", F.to_date("order_ts")))
    silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{S}.orders")
    spark.sql(f"""CREATE OR REPLACE TABLE {G}.daily_sales AS
        SELECT order_date, COUNT(*) orders, SUM(amount) revenue FROM {S}.orders WHERE status='PAID' GROUP BY order_date""")
    spark.sql(f"CREATE OR REPLACE VIEW {G}.v_top_customers AS SELECT customer_id, SUM(amount) total FROM {S}.orders GROUP BY customer_id ORDER BY total DESC LIMIT 10")
    spark.table(f"{G}.daily_sales").show(3)

In [ ]:
with guard("data quality, audit, reconciliation"):
    checks = {
      "no_null_ids": spark.table(f"{S}.orders").filter("order_id IS NULL").count() == 0,
      "no_negative_amount": spark.table(f"{S}.orders").filter("amount < 0").count() == 0,
      "no_dupes": spark.table(f"{S}.orders").count() == spark.table(f"{S}.orders").select("order_id").distinct().count(),
      "row_recon": spark.table(f"{B}.orders").count() == spark.table(f"{S}.orders").count(),
    }
    spark.sql(f"CREATE TABLE IF NOT EXISTS {AD}.audit_log (run_ts TIMESTAMP, check_name STRING, passed BOOLEAN)")
    rows = [(dt.datetime.utcnow(), k, bool(v)) for k, v in checks.items()]
    spark.createDataFrame(rows, "run_ts TIMESTAMP, check_name STRING, passed BOOLEAN").write.mode("append").saveAsTable(f"{AD}.audit_log")
    print(checks); assert all(checks.values()), "quality gate failed"

In [ ]:
with guard("cdc merge and scd type 2"):
    dim = f"{AD}.dim_customer"
    spark.sql(f"""CREATE OR REPLACE TABLE {dim} (customer_id INT, city STRING, valid_from TIMESTAMP, valid_to TIMESTAMP, is_current BOOLEAN)""")
    spark.sql(f"INSERT INTO {dim} SELECT customer_id, city, current_timestamp(), NULL, true FROM {B}.customers")
    chg = spark.createDataFrame([(1, "Mumbai"), (2, "Pune")], "customer_id INT, city STRING")
    chg.createOrReplaceTempView("chg")
    # step 1: close changed current rows
    spark.sql(f"""MERGE INTO {dim} d USING chg c ON d.customer_id=c.customer_id AND d.is_current
                  WHEN MATCHED AND d.city <> c.city THEN UPDATE SET is_current=false, valid_to=current_timestamp()""")
    # step 2: insert new versions for customers without a current row
    spark.sql(f"""INSERT INTO {dim} SELECT c.customer_id, c.city, current_timestamp(), NULL, true FROM chg c
                  LEFT JOIN {dim} d ON d.customer_id=c.customer_id AND d.is_current WHERE d.customer_id IS NULL""")
    spark.sql(f"SELECT * FROM {dim} WHERE customer_id IN (1,2) ORDER BY customer_id, valid_from").show(truncate=False)

In [ ]:
with guard("retry and idempotency"):
    def retry(fn, tries=3, delay=1):
        for i in range(tries):
            try: return fn()
            except Exception as e:
                if i == tries - 1: raise
                time.sleep(delay * 2 ** i)
    retry(lambda: spark.sql(f"SELECT COUNT(*) FROM {S}.orders").collect())
    # idempotent load = MERGE on a business key (re-running yields the same table)

**SCD1** = `MERGE ... WHEN MATCHED THEN UPDATE`. **SCD2** shown above. In pipelines use `AUTO CDC INTO ... SCD TYPE 2` (older names `APPLY CHANGES INTO`, `dlt.apply_changes`). **Expectations** in pipelines: `CONSTRAINT valid_id EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW | FAIL UPDATE`. Late data: use event-time watermarks and MERGE on keys; backfills: rerun with a date parameter and MERGE or `replaceWhere`. Data contracts: enforce with Delta constraints plus schema checks in CI.

## Module 10: Lakeflow Jobs (Workflows)

Jobs are created in the UI (Jobs & Pipelines), via REST (`POST /api/2.2/jobs/create`), CLI (`databricks jobs create --json @job.json`), SDK, or bundles. Task types: notebook, Python script/wheel, SQL, dashboard, pipeline, JAR, Spark Submit, Run Job, If/else, For each. Triggers: scheduled (cron), file arrival, table update, continuous. Retries: `max_retries`, `min_retry_interval_millis`. Pass values between tasks with `dbutils.jobs.taskValues` and `{{tasks.<name>.values.<key>}}`; parameters via `{{job.parameters.<name>}}`. Repair runs re-run only failed tasks.

In [ ]:
job_spec = {
  "name": "learning_etl",
  "parameters": [{"name": "env", "default": "dev"}],
  "tasks": [
    {"task_key": "ingest", "notebook_task": {"notebook_path": "/Workspace/Users/<you>/ingest", "base_parameters": {"env": "{{job.parameters.env}}"}},
     "max_retries": 2, "min_retry_interval_millis": 60000, "timeout_seconds": 3600},
    {"task_key": "check", "depends_on": [{"task_key": "ingest"}],
     "condition_task": {"op": "GREATER_THAN", "left": "{{tasks.ingest.values.rows}}", "right": "0"}},
    {"task_key": "publish", "depends_on": [{"task_key": "check", "outcome": "true"}],
     "sql_task": {"warehouse_id": "<warehouse-id>", "query": {"query_id": "<query-id>"}}}
  ],
  "trigger": {"table_update": {"table_names": ["databricks_learning.bronze.orders"]}},
  "email_notifications": {"on_failure": ["you@example.com"]}
}
print(json.dumps(job_spec, indent=2)[:400], "...")
with guard("jobs list via sdk (read-only)"):
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    print([j.settings.name for j in w.jobs.list(limit=5)])

The `<placeholders>` in `job_spec` are real values you must supply (notebook path, warehouse id, query id); the spec is **not** submitted automatically. Serverless jobs: omit cluster settings; classic: add `job_clusters` + `job_cluster_key`. Permissions: `CAN_VIEW`, `CAN_MANAGE_RUN`, `CAN_MANAGE` via the permissions API or UI.

## Module 12: Performance optimization

In [ ]:
def timed(label, fn):
    t0 = time.time(); r = fn(); print(f"{label}: {time.time()-t0:.2f}s"); return r

with guard("join strategies and AQE"):
    o, c = spark.table(f"{B}.orders"), spark.table(f"{B}.customers")
    timed("sort-merge (hint)", lambda: o.join(c.hint("merge"), "customer_id").count())
    timed("broadcast (hint)", lambda: o.join(F.broadcast(c), "customer_id").count())
    o.join(F.broadcast(c), "customer_id").explain()      # look for BroadcastHashJoin
    for k in ["spark.sql.adaptive.enabled", "spark.sql.adaptive.skewJoin.enabled", "spark.sql.autoBroadcastJoinThreshold"]:
        print(k, "=", conf(k))

In [ ]:
with guard("shuffle partitions before/after"):
    spark.conf.set("spark.sql.shuffle.partitions", "400")
    timed("400 partitions", lambda: o.groupBy("customer_id").count().count())
    spark.conf.set("spark.sql.shuffle.partitions", "8")
    timed("8 partitions", lambda: o.groupBy("customer_id").count().count())

At this data size timings are noise; the **method** is what matters: change one setting, re-run, compare Spark UI stage metrics (duration, shuffle read/write, spill, skew in task-time max vs median). Techniques: broadcast small tables, filter early (predicate pushdown, partition pruning), liquid clustering/Z-order for data skipping, compact small files (`OPTIMIZE`), keep stats fresh (`ANALYZE TABLE`), prefer Photon for SQL/DataFrame workloads, AQE for skew and coalescing, salting only if AQE skew handling is not enough, avoid Python UDFs. `spark.executor.memory` / `spark.driver.memory` are cluster-level settings.

## Module 14: Security and administration

In [ ]:
with guard("system tables (needs system schema access)"):
    spark.sql("SELECT event_time, action_name, user_identity.email FROM system.access.audit ORDER BY event_time DESC LIMIT 5").show(truncate=False)
with guard("billing usage (needs access)"):
    spark.sql("SELECT usage_date, sku_name, SUM(usage_quantity) q FROM system.billing.usage GROUP BY 1,2 ORDER BY 1 DESC LIMIT 5").show()

**Admin-only / non-notebook:** users, groups and SCIM (`/api/2.0/accounts/scim/v2/*`, or identity provider sync); service principals and OAuth M2M (client id/secret in a secret scope, `databricks auth login` for U2M); network security (private link/VPC peering, IP access lists, serverless network policies) **[verify]**; encryption (platform keys or customer-managed keys); secrets ACLs; compute/job/warehouse permissions via UI or permissions API. Enable system schemas with `PUT /api/2.0/unity-catalog/metastores/{id}/systemschemas/{schema}` (metastore/account admin). ABAC policies use governed tags plus policies on catalog/schema/table **[verify, preview in some regions]**.

# LEVEL 4: EXPERT

## Module 11: REST API, CLI and SDK

In [ ]:
with guard("rest api via sdk client"):
    w = WorkspaceClient()
    me = w.api_client.do("GET", "/api/2.0/preview/scim/v2/Me")
    print(me.get("userName"))
    print([c.name for c in w.catalogs.list()][:5])
    print([wh.name for wh in w.warehouses.list()][:5])

**CLI (terminal, not notebook):**
```bash
databricks auth login --host https://<workspace-host>          # OAuth U2M
databricks auth profiles
databricks jobs list --output json
databricks clusters list
databricks warehouses list
databricks catalogs list
databricks secrets create-scope learning
databricks api get /api/2.1/jobs/list
```
**Raw REST (token from a secret, never hardcoded):**
```python
import requests
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.secrets.get("learning", "pat")      # only if you created this scope
r = requests.get(f"{host}/api/2.1/jobs/list", headers={"Authorization": f"Bearer {token}"}, params={"limit": 25})
r.raise_for_status(); data = r.json()               # paginate with next_page_token / page_token
```
Prefer OAuth (service principal M2M) over personal access tokens. Handle `429` with backoff, `401/403` as auth/permission errors. Versions: jobs `2.1`/`2.2`, SCIM `2.0`, Unity Catalog `2.1`, pipelines `2.0`. Account APIs live on `accounts.cloud.databricks.com` (AWS), `accounts.azuredatabricks.net`, `accounts.gcp.databricks.com`.

## Module 13: Machine learning and AI

In [ ]:
with guard("mlflow tracking"):
    import mlflow
    from sklearn.linear_model import LinearRegression
    exp = f"/Users/{ME}/databricks_learning_exp"
    mlflow.set_experiment(exp)
    pdf = spark.table(f"{S}.orders").select("quantity", "amount").toPandas()
    with mlflow.start_run(run_name="lr_demo"):
        m = LinearRegression().fit(pdf[["quantity"]], pdf["amount"])
        mlflow.log_param("model", "linear"); mlflow.log_metric("r2", float(m.score(pdf[["quantity"]], pdf["amount"])))
        mlflow.sklearn.log_model(m, "model")
    print(mlflow.search_runs(experiment_names=[exp]).shape)

**Register in Unity Catalog:** `mlflow.set_registry_uri("databricks-uc")` then `mlflow.register_model("runs:/<run_id>/model", "databricks_learning.sandbox.lr_demo")`; load with `models:/databricks_learning.sandbox.lr_demo@champion`. **Separate products/entitlements:** Model Serving, Vector Search, Foundation Model APIs, AutoML, Feature Engineering in UC, Genie/AI-BI, Agent Framework, Lakehouse Monitoring. Each needs workspace enablement, region support and permissions; they are described, not executed, here **[verify]**. Batch inference: `mlflow.pyfunc.spark_udf(spark, model_uri)`. Hyperparameter tuning: Optuna/Hyperopt (check current Databricks guidance), distributed training: `TorchDistributor`, `spark-tensorflow-distributor`.

## Module 15: Advanced developer techniques

In [ ]:
with guard("testing and metadata-driven pipeline"):
    def clean_orders(df):
        return df.dropDuplicates(["order_id"]).filter("amount >= 0")
    def test_clean_orders():
        t = spark.createDataFrame([(1, 5.0), (1, 5.0), (2, -1.0)], "order_id INT, amount DOUBLE")
        out = clean_orders(t); assert out.count() == 1
    test_clean_orders()
    config = [{"src": f"{B}.customers", "dst": f"{SB}.m_customers", "filter": "city IS NOT NULL"},
              {"src": f"{B}.products",  "dst": f"{SB}.m_products",  "filter": "price > 0"}]
    for c_ in config:
        spark.table(c_["src"]).filter(c_["filter"]).write.mode("overwrite").saveAsTable(c_["dst"])
    import logging; logging.basicConfig(level=logging.INFO); logging.getLogger("etl").info("metadata pipeline ok")

In [ ]:
with guard("write sample bundle file"):
    yml = """bundle:
  name: learning_bundle
variables:
  catalog: {default: databricks_learning_dev}
targets:
  dev:
    mode: development
    default: true
    workspace: {host: https://<dev-workspace-host>}
  prod:
    mode: production
    variables: {catalog: databricks_learning}
    workspace: {host: https://<prod-workspace-host>}
resources:
  jobs:
    etl:
      name: etl_${bundle.target}
      tasks:
        - task_key: run
          notebook_task: {notebook_path: ./etl.py, base_parameters: {catalog: ${var.catalog}}}
"""
    dbutils.fs.put(f"{VOL}/dev/databricks.yml", yml, True)

**CI/CD (terminal/CI):** `databricks bundle init | validate | deploy -t dev | run etl -t dev | summary | destroy -t dev`. Promote one codebase through targets using `variables` and per-target overrides; `mode: development` prefixes resources and pauses schedules, `mode: production` expects explicit permissions and a service-principal `run_as`. Git folders: clone, branch, commit, push, pull and open PRs from the workspace Git UI. Wheels: build with `python -m build`, upload to a volume, install with `%pip install /Volumes/.../pkg.whl`. Terraform: `databricks_catalog`, `databricks_schema`, `databricks_job` resources via the Databricks provider. Pytest runs in CI (`pytest -q`) against Databricks Connect or a bundle-deployed test job. Replace `<...>` hosts before use.

# LEVEL 5: ARCHITECT

## Module 16: Architecture and operations

**Multi-workspace and multi-catalog:** one Unity Catalog metastore per region, many workspaces attached. Catalog-per-environment (`dev/test/prod`) or catalog-per-domain with `MANAGED LOCATION` on separate storage; bind catalogs to workspaces (workspace-catalog binding) to isolate prod. **Multi-tenant:** catalog or schema per tenant plus row filters; separate compute/policies per tenant for isolation and chargeback via tags and `system.billing.usage`.
**Data mesh:** domain-owned catalogs, shared through Delta Sharing/UC grants, central governance (tags, ABAC, lineage). **Federation:** Lakehouse Federation for query-in-place; ingest (Lakeflow Connect/Auto Loader) when latency, load on source or cost matters.
**DR/HA:** Delta data in object storage with cross-region replication or scheduled DEEP CLONE, plus infrastructure as code (Terraform/bundles) to rebuild workspaces; UC metadata needs its own export/replication plan **[verify]**. Test failover regularly.
**Cost governance:** tag compute, use jobs/serverless for production, auto-termination, cluster policies, budgets, `system.billing`. **Large-scale tuning:** right-size files (OPTIMIZE, liquid clustering), Photon, AQE, avoid skew and small files, state-size control in streaming. **CDC at scale:** Change Data Feed or Lakeflow Connect CDC into streaming tables, then `AUTO CDC` for SCD1/2.
**Failure scenarios:** cluster start failure (quota, subnet/IP exhaustion, init script or library errors, cloud capacity); library conflicts (pin versions, use isolated environments); OOM (driver: `collect`/`toPandas`; executor: skew, oversized partitions; fix by repartitioning, filtering, broadcast limits, more memory); streaming stuck (checkpoint/schema change); concurrent write conflicts (partition predicates, retry).
**Trade-offs:** serverless (fast, less control) vs classic (custom configs, libraries); streaming tables (incremental, append-oriented) vs materialized views (managed incremental refresh, SQL-defined) vs views (always current, compute at read) vs tables (full control, you manage refresh).

## Appendix A: Interview Q&A
1. **Delta vs Parquet?** Delta adds a transaction log: ACID, time travel, schema enforcement, MERGE.
2. **VACUUM vs OPTIMIZE?** OPTIMIZE compacts files; VACUUM deletes unreferenced files older than retention (default 7 days).
3. **Managed vs external table?** Managed: UC owns data lifecycle; dropping deletes data. External: you own the files; dropping keeps them.
4. **Auto Loader vs COPY INTO?** Auto Loader scales to very many files and supports schema evolution; COPY INTO is simple, idempotent, SQL-only, for smaller file counts.
5. **Narrow vs wide transformation?** Wide needs a shuffle and starts a new stage.
6. **repartition vs coalesce?** repartition shuffles to any count; coalesce only reduces partitions without a full shuffle.
7. **View vs materialized view vs streaming table?** See Module 16 trade-offs.
8. **How do you handle skew?** AQE skew join, broadcast, salting, better keys.
9. **Exactly-once in streaming?** Replayable source + checkpoint + idempotent/transactional sink.
10. **Shallow vs deep clone?** Shallow copies metadata only; deep copies data.

## Appendix B: Glossary
**Lakehouse**, data lake plus warehouse features. **Unity Catalog**, central governance. **Metastore**, top UC container per region. **Volume**, UC-governed file storage. **DBU**, Databricks billing unit. **Photon**, native vectorized engine. **AQE**, adaptive query execution. **Liquid clustering**, adaptive data layout replacing partitioning/Z-order. **CDF**, change data feed. **SCD**, slowly changing dimension. **Medallion**, bronze/silver/gold layering. **Lakeflow**, umbrella for Connect, Spark Declarative Pipelines (formerly DLT) and Jobs (formerly Workflows). **Declarative Automation Bundles**, formerly Databricks Asset Bundles.

## Appendix C: Deprecated to modern
`dbutils.library` to `%pip`; DBFS mounts to volumes/external locations; Delta Live Tables to Lakeflow Spark Declarative Pipelines; `APPLY CHANGES INTO` to `AUTO CDC INTO`; `trigger(once=True)` to `availableNow=True`; Repos to Git folders; Workflows to Lakeflow Jobs; Asset Bundles to Declarative Automation Bundles; Hive metastore tables to Unity Catalog.

## Appendix D: Command index (documented-structure entries)
- M2: `dbutils.fs`
- M2: `dbutils.widgets`
- M2: `dbutils.secrets`

The full searchable list is in `Databricks_Command_Reference.md`. Feature compatibility is in `Databricks_Compatibility_Matrix.md`.

## Final validation and summary

In [ ]:
def validate_all():
    out = {}
    for t in need + [f"{S}.orders", f"{G}.daily_sales", f"{SB}.delta_demo", f"{AD}.dim_customer", f"{AD}.audit_log"]:
        out[f"table {t}"] = spark.catalog.tableExists(t) and spark.table(t).limit(1).count() >= 0
    out["volume files"] = len(dbutils.fs.ls(f"{VOL}/raw")) > 0
    out["delta history"] = spark.sql(f"DESCRIBE HISTORY {SB}.delta_demo").count() > 0
    out["sql exec"] = spark.sql("SELECT 1").first()[0] == 1
    out["pyspark exec"] = spark.range(5).count() == 5
    return out

for k, v in validate_all().items():
    print(("OK   " if v else "MISS ") + k)
print()
from collections import Counter
print(Counter(r[1] for r in RESULTS))
for name, status, why in RESULTS:
    if status != "PASSED": print(f"{status:8} {name}  {why}")
print("\nAdmin actions, if anything failed: see Setup notebook section 4 (CREATE CATALOG grant) and section 10.")